# Lasso Regression: Sparse Models for Disease Progression

**Family:** Supervised learning | **Level:** Intermediate | **Dataset:** `diabetes` | **Runs on:** CPU, about 1 minute(s)

Use the L1 penalty to set coefficients to exactly zero and select features.

## Table of Contents

1. [Learning Objectives](#1.-Learning-Objectives)
2. [Real-World Problem](#2.-Real-World-Problem)
3. [Research Paper](#3.-Research-Paper)
4. [Mathematical Foundation](#4.-Mathematical-Foundation)
5. [Dataset Introduction](#5.-Dataset-Introduction)
6. [Exploratory Data Analysis](#6.-Exploratory-Data-Analysis)
7. [Data Preprocessing](#7.-Data-Preprocessing)
8. [Algorithm Implementation](#8.-Algorithm-Implementation)
9. [Model Training](#9.-Model-Training)
10. [Evaluation](#10.-Evaluation)
11. [Visualization](#11.-Visualization)
12. [Interpretation](#12.-Interpretation)
13. [Experiments and Challenges](#13.-Experiments-and-Challenges)
14. [Research Discussion](#14.-Research-Discussion)
15. [Summary and Further Reading](#15.-Summary-and-Further-Reading)

## 1. Learning Objectives

**After this notebook you will understand:**

- How the L1 penalty creates sparse models
- How coordinate descent and soft thresholding work
- How to count selected features along the path

**Prerequisites:**

- Linear regression notebook
- Train, validation and test splits

**Skills you will have afterwards:**

- Choose a hyperparameter on validation data
- Compare with a baseline
- Run a sensitivity experiment

## 2. Real-World Problem

**Context.** Clinical teams prefer a score built from few measurements, because each measurement costs time and money.

**The problem.** Predict diabetes progression with as few baseline features as possible.

**Why it matters.** A short list of predictors is easier to collect, audit and explain.

**Why machine learning.** The relevant subset of markers is not known in advance.

**Why this algorithm.** Lasso fits and selects in one step.

## 3. Research Paper

Entries are grouped by type. Bibliographic details marked *verified* were checked against the publisher record (Crossref) or the arXiv abstract page on 2026-11-17. Entries marked *not verified* come from the author's knowledge and must be checked before citing.

### Regression Shrinkage and Selection Via the Lasso

- **Type:** foundational paper
- **Authors:** Tibshirani, R.
- **Year and venue:** 1996, Journal of the Royal Statistical Society B 58(1):267-288
- **DOI or URL:** https://doi.org/10.1111/j.2517-6161.1996.tb02080.x
- **Verification:** Crossref 2026-11-17
- **Research objective:** Propose L1 penalised least squares for shrinkage and selection.
- **Methodology:** Minimise squared error subject to a bound on the sum of absolute coefficients.
- **Dataset:** Prostate cancer data and simulations.
- **Main findings:** Produces sparse, interpretable models and often better prediction than subset selection.
- **Limitations:** Picks one variable among correlated ones. Selection is unstable.

### Least angle regression

- **Type:** foundational paper
- **Authors:** Efron, B., Hastie, T., Johnstone, I. and Tibshirani, R.
- **Year and venue:** 2004, The Annals of Statistics 32(2):407-499
- **DOI or URL:** https://doi.org/10.1214/009053604000000067
- **Verification:** Crossref 2026-11-17
- **Research objective:** Give an efficient path algorithm for the lasso and related methods.
- **Methodology:** Move coefficients along the direction equiangular to the active predictors.
- **Dataset:** The diabetes progression data of 442 patients used in this repository.
- **Main findings:** The full lasso path costs about the same as one least squares fit.
- **Limitations:** Sensitive to noise when predictors are strongly correlated.

**Relationship to this notebook.** Tibshirani introduces the lasso and Efron et al. give the path algorithm and analyse these same 442 patients. Our notebook uses a different split and software, so selected sets may differ. This notebook does not reproduce the experimental protocol of any listed paper. It uses a different dataset and split, so numbers are not comparable with the paper.

## 4. Mathematical Foundation

**Objective.**

$$\hat\beta^{\text{lasso}} = \arg\min_\beta \; \frac{1}{2n}\lVert y - A\beta \rVert_2^2 + \alpha \lVert \beta \rVert_1$$

(scikit-learn scales the squared error by $1/2n$.)

**Soft thresholding.** For one standardised feature the solution is $\hat\beta_j = \operatorname{sign}(z_j)\,\max(|z_j| - \alpha, 0)$, where $z_j$ is the least squares estimate. Small coefficients become exactly zero.

**Optimisation.** Coordinate descent updates one coefficient at a time using the soft threshold.

**Assumptions.** Sparse truth, standardised features.

**Limitations.** Among correlated features it tends to pick one and drop the others, and the selected set can change with the sample.

## 5. Dataset Introduction

**Diabetes progression** from Efron, Hastie, Johnstone and Tibshirani (2004), shipped with scikit-learn. 442 patients, 10 baseline features (`age`, `sex`, `bmi`, `bp`, and six blood serum measures `s1` to `s6`), target is a quantitative measure of disease progression one year after baseline. The provider centred and scaled the features. Data terms of the original source are not stated by the provider. Limits: small sample, no time dimension, `sex` is a standardised number.

The table printed below the next cell comes from the dataset registry in `src/ml_lab/data/registry.py`. Licence terms belong to the data provider, not to this repository.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_validate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from ml_lab.data import load_dataset, missing_report
from ml_lab.preprocessing import split_three_way, build_preprocessor, make_model_pipeline, feature_names
from ml_lab.evaluation import evaluate_estimator, compare_models
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz

NOTEBOOK_ID = "lasso_regression"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

In [ ]:
bundle = load_dataset("diabetes")
data = bundle.frame
TASK = bundle.task
TARGET = bundle.target
print("task:", TASK, "| target:", TARGET, "| shape:", data.shape)
display(pd.Series(bundle.describe_source(), name="source").to_frame())
display(data.head())

## 6. Exploratory Data Analysis

We look at size, types, missing values, the target, feature distributions, correlation and outliers before any modelling. 

In [ ]:
print("Rows and columns:", data.shape)
display(data.dtypes.value_counts().rename("columns by dtype").to_frame())
display(data.describe().T.round(3).head(20))
missing = missing_report(data)
print("Columns with missing values:", len(missing))
display(missing)
print("Duplicate rows:", int(data.duplicated().sum()))

In [ ]:
fig = viz.plot_target(data[TARGET], TASK)
plt.show()
fig = viz.plot_distributions(data.drop(columns=[TARGET]))
plt.show()
fig = viz.plot_correlation(data)
plt.show()
display(viz.outlier_table(data.drop(columns=[TARGET])).head(8))

## 7. Data Preprocessing

Rules used here:

- Split first, then fit imputers, scalers and encoders on the training rows only. This prevents data leakage.
- The test set is touched once, at the end.
- Order inside the pipeline: impute, scale or encode, then model.


In [ ]:

X, y = data.drop(columns=[TARGET]), data[TARGET]
X_train, X_val, X_test, y_train, y_val, y_test = split_three_way(X, y, task=TASK, seed=SEED)
print("train / validation / test rows:", len(X_train), len(X_val), len(X_test))
if TASK == "classification":
    split_rates = pd.DataFrame({"train": y_train.value_counts(normalize=True),
                                "validation": y_val.value_counts(normalize=True),
                                "test": y_test.value_counts(normalize=True)}).round(3)
    display(split_rates)
prep = build_preprocessor(X_train, scale=True, encode="onehot")
Xtr = prep.fit_transform(X_train)   # fit on train only
Xva, Xte = prep.transform(X_val), prep.transform(X_test)
print("transformed train shape:", Xtr.shape)

## 8. Algorithm Implementation

### 8.1 Concept

The L1 ball has corners on the axes, so the constrained optimum often lies where some coefficients are zero.

### 8.2 Python implementation from scratch

We implement cyclic coordinate descent with soft thresholding and check it against scikit-learn.

In [ ]:
ytr = y_train.values

def soft_threshold(z, t):
    return np.sign(z) * np.maximum(np.abs(z) - t, 0)

def lasso_cd(X, y, alpha, sweeps=300):
    n, p = X.shape
    w, b = np.zeros(p), y.mean()
    for _ in range(sweeps):
        for j in range(p):
            residual = y - b - X @ w + X[:, j] * w[j]
            w[j] = soft_threshold(X[:, j] @ residual / n, alpha) / (X[:, j] @ X[:, j] / n)
        b = np.mean(y - X @ w)
    return w, b

w_mine, b_mine = lasso_cd(Xtr, ytr, alpha=1.0)
reference = Lasso(alpha=1.0, max_iter=10000).fit(Xtr, ytr)
print("max coefficient gap vs sklearn:", np.abs(w_mine - reference.coef_).max())
print("non zero coefficients (scratch, sklearn):", int((w_mine != 0).sum()), int((reference.coef_ != 0).sum()))

### 8.3 Library implementation

The estimator is wrapped in a pipeline so imputation, scaling and encoding are fitted on training rows only.

### 8.4 Important parameters

| Parameter | Meaning |
|---|---|
| `alpha` | L1 strength. Larger means fewer features. |
| `max_iter` | Coordinate descent sweeps. Raise if a convergence warning appears. |

## 9. Model Training

Procedure: fit on the training rows, choose one hyperparameter on the validation rows, refit on training plus validation, then evaluate once on the test rows. The random seed is fixed (`SEED = 42`). 

In [ ]:
SELECT = "RMSE"
LOWER_IS_BETTER = True
SCORING = "neg_root_mean_squared_error"
SWEEP_PARAM = "model__alpha"
SWEEP_VALUES = [0.001, 0.01, 0.1, 0.3, 1, 3, 10]

def build_pipeline():
    return make_model_pipeline(X_train, Lasso(alpha=0.1, max_iter=10000), scale=True, encode="onehot")

pipe = build_pipeline()
print(pipe.named_steps["model"])
rows = []
for value in SWEEP_VALUES:
    candidate = clone(pipe).set_params(**{SWEEP_PARAM: value}).fit(X_train, y_train)
    rows.append({SWEEP_PARAM: value, **evaluate_estimator(candidate, X_val, y_val, TASK)})
sweep = pd.DataFrame(rows).set_index(SWEEP_PARAM)
display(sweep[[c for c in [SELECT, "R2", "MAE", "Accuracy", "F1", "ROC_AUC", "PR_AUC"] if c in sweep.columns]].round(4))
best_value = sweep[SELECT].idxmin() if LOWER_IS_BETTER else sweep[SELECT].idxmax()
print("best", SWEEP_PARAM, "on validation:", best_value)
final_model = clone(pipe).set_params(**{SWEEP_PARAM: best_value})
X_trainval, y_trainval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
final_model.fit(X_trainval, y_trainval)   # refit before test
print("refit on train plus validation rows")

## 10. Evaluation

Regression metrics: MAE (average absolute error in target units), RMSE (square root of the mean squared error, same unit, punishes large errors), R-squared (share of variance explained, can be negative on new data) and MAPE when the target is never zero.

A dummy model gives the floor every real model must beat.

In [ ]:
baseline = DummyRegressor(strategy="mean") if TASK == "regression" else DummyClassifier(strategy="prior")
baseline = make_model_pipeline(X_train, baseline, scale=False).fit(X_trainval, y_trainval)
test_metrics = evaluate_estimator(final_model, X_test, y_test, TASK)
baseline_metrics = evaluate_estimator(baseline, X_test, y_test, TASK)
print("Test set, scored once after model choice")
display(compare_models({"baseline": baseline_metrics, "Lasso": test_metrics}))
cv = cross_validate(final_model, X_trainval, y_trainval, cv=5, scoring=SCORING, n_jobs=1)
cv_scores = cv["test_score"]
print(f"5-fold CV {SCORING}: mean {cv_scores.mean():.4f}, std {cv_scores.std(ddof=1):.4f}")
record = ExperimentRecord(notebook=NOTEBOOK_ID, family="supervised", algorithm="Lasso", dataset="diabetes",
                          task=TASK, seed=SEED, config={"param": SWEEP_PARAM, "value": str(best_value)},
                          metrics={k: float(v) for k, v in test_metrics.items()},
                          extra={"baseline": {k: float(v) for k, v in baseline_metrics.items()},
                                 "cv_mean": float(cv_scores.mean()), "cv_std": float(cv_scores.std(ddof=1)),
                                 "cv_scoring": SCORING, "select_metric": SELECT})
print("saved", record.save().name)

## 11. Visualization

All plots use the test rows unless the title says otherwise.

In [ ]:
pred = final_model.predict(X_test)
fig = viz.plot_pred_vs_actual(y_test, pred, unit="(progression score)")
save_figure(fig, NOTEBOOK_ID, "pred_vs_actual"); plt.show()
fig = viz.plot_residuals(y_test, pred, unit="(progression score)")
save_figure(fig, NOTEBOOK_ID, "residuals"); plt.show()
fig = viz.plot_learning_curve(final_model, X_trainval, y_trainval, scoring=SCORING)
save_figure(fig, NOTEBOOK_ID, "learning_curve"); plt.show()

## 12. Interpretation

Permutation importance shuffles one feature and measures how much the error grows. Correlated features share importance, so read it as a ranking of groups. Check the residual plot for curvature or funnel shapes, which mean the model misses structure or the error variance changes with the prediction.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring=SCORING, n_repeats=10, random_state=SEED)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
fig = viz.plot_importance(importance.index, importance.values, title="Permutation importance on the test set",
                          top=12, xlabel="Drop in " + SCORING + " when shuffled")
save_figure(fig, NOTEBOOK_ID, "importance"); plt.show()
print("Top features:", ", ".join(importance.index[:5]))
errors = (y_test.values - final_model.predict(X_test)) if TASK == "regression" else (y_test.values != final_model.predict(X_test))
if TASK == "regression":
    worst = pd.DataFrame({"actual": y_test.values, "predicted": final_model.predict(X_test)}, index=y_test.index)
    worst["abs_error"] = (worst["actual"] - worst["predicted"]).abs()
    print("Largest test errors:")
    display(worst.sort_values("abs_error", ascending=False).head(5).round(3))
else:
    print("Misclassified test rows:", int(errors.sum()), "of", len(errors))
print()
print(f"Test {SELECT}: {test_metrics[SELECT]:.4f} | baseline {SELECT}: {baseline_metrics[SELECT]:.4f}")
print(f"Cross validation spread (std): {cv_scores.std(ddof=1):.4f}. A test gap larger than this spread is meaningful.")

## 13. Experiments and Challenges

### Experiment 1: Number of selected features along the path

Count non zero coefficients as alpha grows.

In [ ]:
alphas = np.logspace(-3, 1.5, 40)
counts = [int((Lasso(alpha=a, max_iter=10000).fit(Xtr, y_train).coef_ != 0).sum()) for a in alphas]
plt.semilogx(alphas, counts, marker="o"); plt.xlabel("alpha"); plt.ylabel("Non zero coefficients")
plt.title("Sparsity versus penalty strength"); plt.show()

### Experiment 2: Selection stability under resampling

Fit lasso on 30 bootstrap samples and count how often each feature is selected. Features chosen in most samples are more trustworthy.

In [ ]:
rng = np.random.default_rng(SEED)
chosen = np.zeros(Xtr.shape[1])
for _ in range(30):
    idx = rng.integers(0, len(Xtr), len(Xtr))
    chosen += Lasso(alpha=1.0, max_iter=10000).fit(Xtr[idx], y_train.values[idx]).coef_ != 0
freq = pd.Series(chosen / 30, index=X_train.columns).sort_values(ascending=False)
display(freq.round(2).to_frame("selection frequency"))

### Experiment 3: Lasso versus ridge on prediction

Compare cross validated RMSE of the best lasso and ridge alphas.

In [ ]:
from sklearn.model_selection import cross_val_score
def cv_rmse(model, X_data=None, y_data=None):
    scores = cross_val_score(model, X_data if X_data is not None else X_trainval,
                             y_data if y_data is not None else y_trainval,
                             cv=5, scoring="neg_root_mean_squared_error")
    return -scores.mean(), scores.std(ddof=1)
from sklearn.linear_model import Ridge
for name, model in [("Lasso alpha=1", Lasso(alpha=1.0, max_iter=10000)), ("Ridge alpha=10", Ridge(alpha=10))]:
    mean, std = cv_rmse(make_model_pipeline(X_trainval, model))
    print(f"{name}: CV RMSE {mean:.2f} +- {std:.2f}")

### Exercises for learners

1. Repeat the notebook with `SEED = 7` and report how much the headline metric changes.
2. Add one more feature or remove one and explain the change using cross validation.
3. Replace the validation sweep with `GridSearchCV` and compare the chosen value.

## 14. Research Discussion

Tibshirani describes sparse solutions and Efron et al. show the piecewise linear path on the diabetes patients. Our coefficient counts plot shows the same qualitative path. We make no claim about matching their selected variable list.

**Dataset and method differences.** The papers above use other datasets, other splits and often other implementations. We do not claim that our numbers agree or disagree with them. What this notebook can show is whether the behaviour described in the paper (for example the effect of a parameter) appears on our data.

**Possible improvements.** Use `LassoCV` with repeated cross validation, stability selection, or the relaxed lasso to reduce shrinkage bias on kept features.

**Research limitations.** One dataset, one split seed for the headline number, and small test sets for the UCI data. Cross validation spread is reported to show how unstable a single split can be.

## 15. Summary and Further Reading

**Key takeaways**

- L1 gives exact zeros.
- Selection can be unstable across samples.
- Standardise before penalising.

**Use cases.** High dimensional data, feature screening, interpretable scores.

**Limitations.** Arbitrary pick among correlated features.

**Recommended next notebook.** Elastic Net.

**Further reading.** See `references.md` in this folder and the library documentation for the classes used above.